## Two-Tower 500k — Colab GPU Training

**Before running:**
1. Runtime > Change runtime type > L4 GPU
2. Upload service account JSON when Cell 2 prompts you
3. Expected time: ~40–60 min for 20 epochs on T4


In [1]:
%%capture
!pip install faiss-gpu google-cloud-bigquery google-cloud-storage \
             gcsfs pyarrow pandas torch --quiet


In [2]:
!git clone -b dev https://github.com/sbnikhil/RecoSys.git
!cd RecoSys

fatal: destination path 'RecoSys' already exists and is not an empty directory.


In [4]:
from google.colab import files
import os

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No file uploaded")

# Use the real filename from the upload
fname = next(iter(uploaded.keys()))
path = os.path.abspath(fname)
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = path
print("GOOGLE_APPLICATION_CREDENTIALS =", path)


Saving recosys-service-account.json to recosys-service-account.json
GOOGLE_APPLICATION_CREDENTIALS = /content/recosys-service-account.json


In [5]:
import sys
%cd /content/RecoSys
sys.path.insert(0, '/content/RecoSys')
!ls


/content/RecoSys
artifacts  notebooks  reports		scripts
LICENSE    README.md  requirements.txt	src


In [6]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("WARNING: No GPU. Go to Runtime > Change runtime type > T4 GPU")


Device: cuda
GPU: NVIDIA A100-SXM4-40GB
Memory: 42.4 GB


In [6]:
# Upload the artifacts/500k/ folder contents from your local machine.
# Easiest: zip it locally, upload here, unzip.
# Run this on your LOCAL machine first:
#   cd artifacts && zip -r 500k_artifacts.zip 500k/
# Then upload the zip here:

from google.colab import files
import zipfile

print("Upload 500k_artifacts.zip:")
uploaded = files.upload()
with zipfile.ZipFile("500k_artifacts.zip", "r") as z:
    z.extractall("artifacts/")
print("Artifacts extracted:")
!ls artifacts/500k/


Upload 500k_artifacts.zip:


Saving 500k_artifacts.zip to 500k_artifacts.zip
Artifacts extracted:
ls: cannot access 'artifacts/500k/': No such file or directory


In [7]:
with zipfile.ZipFile("500k.zip", "r") as z:
    z.extractall("artifacts/")
print("Artifacts extracted:")
!ls artifacts/500k/

Artifacts extracted:
checkpoints	   items_encoded.parquet  user_scalers.pkl
eval_results.json  train_pairs.parquet	  users_encoded.parquet
item_scalers.pkl   train_raw		  vocabs.pkl


In [7]:
import pickle
import pandas as pd

ARTIFACTS_DIR = "artifacts/500k/"
GCS_TEST_PATH = "gs://recosys-data-bucket/samples/users_sample_500k/test/"

with open(f"{ARTIFACTS_DIR}vocabs.pkl", "rb") as f:
    vocabs = pickle.load(f)

items_enc   = pd.read_parquet(f"{ARTIFACTS_DIR}items_encoded.parquet")
users_enc   = pd.read_parquet(f"{ARTIFACTS_DIR}users_encoded.parquet")
train_pairs = pd.read_parquet(f"{ARTIFACTS_DIR}train_pairs.parquet")
test_df     = pd.read_parquet(GCS_TEST_PATH)

print(f"items_encoded : {items_enc.shape}")
print(f"users_encoded : {users_enc.shape}")
print(f"train_pairs   : {train_pairs.shape}")
print(f"test_df       : {test_df.shape}")
print(f"n_users       : {len(vocabs['user2idx']):,}")
print(f"n_items       : {len(vocabs['item2idx']):,}")


items_encoded : (284523, 9)
users_encoded : (445150, 12)
train_pairs   : (7473130, 3)
test_df       : (3451452, 9)
n_users       : 445,150
n_items       : 284,523


## Config
Edit the cell below before training. USE_CONFIDENCE_WEIGHTING=False
based on 50k experiments (see reports/05_model_experiments_50k.md).


In [ ]:
# ── Hyperparameters ──────────────────────────────────────
N_EPOCHS                 = 30
BATCH_SIZE               = 2048
LEARNING_RATE            = 1e-3
WEIGHT_DECAY             = 1e-5
TEMPERATURE              = 0.05
EVAL_EVERY               = 5
USE_CONFIDENCE_WEIGHTING = False
CHECKPOINT_DIR           = "artifacts/500k/checkpoints/"

# Trained items filter — critical, scope FAISS to trained items only
TRAINED_ITEM_IDXS = set(train_pairs.item_idx.unique())
print(f"Trained items : {len(TRAINED_ITEM_IDXS):,} of "
      f"{len(vocabs['item2idx']):,} total")
print(f"Config ready.")


Trained items : 201,648 of 284,523 total
Config ready.


In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 112.5 MB/s eta 0:00:00


In [ ]:
import functools
from src.two_tower.models.two_tower import UserTower, ItemTower, TwoTowerModel
from src.two_tower.training.train import train
from src.two_tower.evaluation.evaluate import evaluate

user_tower = UserTower(
    n_users    = len(vocabs['user2idx']),
    n_top_cats = len(vocabs['top_cat_vocab']),
)
item_tower = ItemTower(
    n_items  = len(vocabs['item2idx']),
    n_cat_l1 = len(vocabs['cat_l1_vocab']),
    n_cat_l2 = len(vocabs['cat_l2_vocab']),
    n_brands = len(vocabs['brand_vocab']),
)
model = TwoTowerModel(user_tower, item_tower, temperature=TEMPERATURE)
model.model_summary()

eval_callback = functools.partial(
    evaluate,
    items_encoded_df  = items_enc,
    users_encoded_df  = users_enc,
    test_df           = test_df,
    train_pairs_df    = train_pairs,
    vocabs            = vocabs,
    device            = torch.device(device),
    k                 = 10,
    trained_item_idxs = TRAINED_ITEM_IDXS,
)

history = train(
    model                    = model,
    train_pairs_df           = train_pairs,
    users_encoded_df         = users_enc,
    items_encoded_df         = items_enc,
    n_epochs                 = N_EPOCHS,
    batch_size               = BATCH_SIZE,
    learning_rate            = LEARNING_RATE,
    weight_decay             = WEIGHT_DECAY,
    device_str               = device,
    checkpoint_dir           = CHECKPOINT_DIR,
    eval_every               = EVAL_EVERY,
    eval_fn                  = eval_callback,
    use_confidence_weighting = USE_CONFIDENCE_WEIGHTING,
)

TwoTowerModel — parameter summary
  User tower :   14,280,504
  Item tower :    9,224,864
  Total      :   23,505,368
Training on cuda — 30 epochs, batch 2048, lr 0.001 → 1e-05 (cosine), wd 1e-05
  Dataset size              : 7,473,130 pairs
  Batches/epoch             : 3,649
  Confidence weighting      : False
TwoTowerModel — parameter summary
  User tower :   14,280,504
  Item tower :    9,224,864
  Total      :   23,505,368

Epoch 1/30
  batch 100/3649 — loss: 7.3166
  batch 200/3649 — loss: 7.1101
  batch 300/3649 — loss: 7.0534
  batch 400/3649 — loss: 6.9033
  batch 500/3649 — loss: 6.7891
  batch 600/3649 — loss: 6.7174
  batch 700/3649 — loss: 6.6676
  batch 800/3649 — loss: 6.5720
  batch 900/3649 — loss: 6.5719
  batch 1000/3649 — loss: 6.5368
  batch 1100/3649 — loss: 6.5203
  batch 1200/3649 — loss: 6.3887
  batch 1300/3649 — loss: 6.3972
  batch 1400/3649 — loss: 6.4392
  batch 1500/3649 — loss: 6.3866
  batch 1600/3649 — loss: 6.2751
  batch 1700/3649 — loss: 6.3372
  ba

In [ ]:
from google.cloud import storage

def upload_to_gcs(local_path, blob_path,
                  bucket_name="recosys-data-bucket"):
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob   = bucket.blob(blob_path)
    blob.upload_from_filename(local_path)
    print(f"Uploaded → gs://{bucket_name}/{blob_path}")

# Upload all checkpoints
import os
ckpt_files = sorted(os.listdir(CHECKPOINT_DIR))
for f in ckpt_files:
    upload_to_gcs(
        f"{CHECKPOINT_DIR}{f}",
        f"models/two_tower_500k/{f}"
    )

# Upload artifacts
upload_to_gcs(f"{ARTIFACTS_DIR}vocabs.pkl",
              "models/two_tower_500k/vocabs_500k.pkl")

print("All checkpoints and vocabs saved to GCS.")
print("Training complete — share Recall@10 results.")


Uploaded → gs://recosys-data-bucket/models/two_tower_500k/epoch_1.pt
Uploaded → gs://recosys-data-bucket/models/two_tower_500k/epoch_10.pt
Uploaded → gs://recosys-data-bucket/models/two_tower_500k/epoch_11.pt
Uploaded → gs://recosys-data-bucket/models/two_tower_500k/epoch_12.pt
Uploaded → gs://recosys-data-bucket/models/two_tower_500k/epoch_13.pt
Uploaded → gs://recosys-data-bucket/models/two_tower_500k/epoch_14.pt
Uploaded → gs://recosys-data-bucket/models/two_tower_500k/epoch_15.pt
Uploaded → gs://recosys-data-bucket/models/two_tower_500k/epoch_16.pt
Uploaded → gs://recosys-data-bucket/models/two_tower_500k/epoch_17.pt
Uploaded → gs://recosys-data-bucket/models/two_tower_500k/epoch_18.pt
Uploaded → gs://recosys-data-bucket/models/two_tower_500k/epoch_19.pt
Uploaded → gs://recosys-data-bucket/models/two_tower_500k/epoch_2.pt
Uploaded → gs://recosys-data-bucket/models/two_tower_500k/epoch_20.pt
Uploaded → gs://recosys-data-bucket/models/two_tower_500k/epoch_21.pt
Uploaded → gs://recosy

## Diagnostics:

In [ ]:
from google.cloud import storage
import os

def download_from_gcs(bucket_name, blob_path, local_path):
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(blob_path)
    blob.download_to_filename(local_path)
    print(f"Downloaded -> {local_path}")

os.makedirs("artifacts/500k/checkpoints", exist_ok=True)
download_from_gcs("recosys-data-bucket", "models/two_tower_500k/epoch_5.pt", "artifacts/500k/checkpoints/epoch_5.pt")
download_from_gcs("recosys-data-bucket", "models/two_tower_500k/epoch_30.pt", "artifacts/500k/checkpoints/epoch_30.pt")

Downloaded -> artifacts/500k/checkpoints/epoch_5.pt
Downloaded -> artifacts/500k/checkpoints/epoch_30.pt


Training curve plot:

In [ ]:
!python scripts/two_tower/plot_training_curves.py

  500k Two-Tower — Loss vs Recall@10 correlation
  Epochs evaluated   : [5, 10, 15, 20, 25, 30]
  Train losses       : [5.0083, 4.6558, 4.4835, 4.3672, 4.2882, 4.2533]
  Recall@10 values   : [0.0088, 0.0086, 0.008, 0.0082, 0.0083, 0.0083]
  Best Recall@10     : 0.0088  (epoch 5)

  Pearson r (loss vs recall) : +0.7701
  Interpretation : NEAR-ZERO / POSITIVE correlation — DECOUPLING.
                   Training objective is NOT driving recall gains.
                   Improving loss further will not help retrieval.

  Figure saved to: /content/RecoSys/reports/figures/500k_training_curves.png


Embedding diagnostics on both checkpoints

In [ ]:
!python scripts/two_tower/diagnose_embeddings.py --checkpoint artifacts/500k/checkpoints/epoch_5.pt

Artifacts dir : /content/RecoSys/artifacts/500k
Loading artifacts...
  items_encoded  : (284523, 9)
  users_encoded  : (445150, 12)
  device         : cuda

Loading checkpoint: epoch_5.pt ...
  Epoch      : 5
  Train loss : 5.0083

Sampling 3,000 users  and  3,000 items  (seed=42)
Encoding user sample through user tower ...
Encoding item sample through item tower ...
Encoding cross pairs ...
Computing similarity matrices ...

  EMBEDDING COLLAPSE DIAGNOSTIC
  Checkpoint : epoch_5.pt
  Epoch      : 5   |   Train loss : 5.0083
  Sample size: 3,000   |   Seed: 42

  [1] USER-USER PAIRWISE COSINE SIMILARITY
      (3,000 users, 8,997,000 off-diagonal pairs)
      Healthy range: mean 0.01 – 0.10  |  Collapse threshold: > 0.30

  Statistic            Value
  --------------  ----------
  mean              0.141817
  std               0.181473
  p10              -0.090828
  p50               0.139652
  p90               0.377442

  OK               : mean 0.1418 is below threshold 0.30

  -----

In [ ]:
!python scripts/two_tower/diagnose_embeddings.py --checkpoint artifacts/500k/checkpoints/epoch_30.pt

Artifacts dir : /content/RecoSys/artifacts/500k
Loading artifacts...
  items_encoded  : (284523, 9)
  users_encoded  : (445150, 12)
  device         : cuda

Loading checkpoint: epoch_30.pt ...
  Epoch      : 30
  Train loss : 4.2533

Sampling 3,000 users  and  3,000 items  (seed=42)
Encoding user sample through user tower ...
Encoding item sample through item tower ...
Encoding cross pairs ...
Computing similarity matrices ...

  EMBEDDING COLLAPSE DIAGNOSTIC
  Checkpoint : epoch_30.pt
  Epoch      : 30   |   Train loss : 4.2533
  Sample size: 3,000   |   Seed: 42

  [1] USER-USER PAIRWISE COSINE SIMILARITY
      (3,000 users, 8,997,000 off-diagonal pairs)
      Healthy range: mean 0.01 – 0.10  |  Collapse threshold: > 0.30

  Statistic            Value
  --------------  ----------
  mean              0.050014
  std               0.168687
  p10              -0.166811
  p50               0.049717
  p90               0.266680

  OK               : mean 0.0500 is below threshold 0.30

  -

Score distribution diagnostic on both checkpoints

In [ ]:
!python scripts/two_tower/diagnose_scores.py --checkpoint artifacts/500k/checkpoints/epoch_5.pt --test-path {GCS_TEST_PATH}

  Score Distribution Diagnostic
  Checkpoint   : epoch_5.pt
  Artifacts dir: /content/RecoSys/artifacts/500k
  Test path    : gs://recosys-data-bucket/samples/users_sample_500k/test/
  Device       : cuda

Loading artifacts ...
  items_encoded  : (284523, 9)
  users_encoded  : (445150, 12)
  train_pairs    : (7473130, 3)

Loading test split from gs://recosys-data-bucket/samples/users_sample_500k/test/ ...
  test_df        : (3451452, 9)

Loading checkpoint: epoch_5.pt ...
  Epoch      : 5
  Train loss : 5.0083

  PART A — TRAINING SCORE DISTRIBUTION
  5 batches × 2,048 pairs  (raw cosine sim, pre-temperature)
  Sampling 5 batches of 2,048 pairs from 7,473,130 training pairs ...
    batch 1/5  |  pos mean: 0.3790  |  neg mean: 0.0513
    batch 2/5  |  pos mean: 0.3817  |  neg mean: 0.0548
    batch 3/5  |  pos mean: 0.3783  |  neg mean: 0.0531
    batch 4/5  |  pos mean: 0.3735  |  neg mean: 0.0513
    batch 5/5  |  pos mean: 0.3747  |  neg mean: 0.0521

  ------------------------------

In [ ]:
!python scripts/two_tower/diagnose_scores.py --checkpoint artifacts/500k/checkpoints/epoch_30.pt --test-path {GCS_TEST_PATH}

  Score Distribution Diagnostic
  Checkpoint   : epoch_30.pt
  Artifacts dir: /content/RecoSys/artifacts/500k
  Test path    : gs://recosys-data-bucket/samples/users_sample_500k/test/
  Device       : cuda

Loading artifacts ...
  items_encoded  : (284523, 9)
  users_encoded  : (445150, 12)
  train_pairs    : (7473130, 3)

Loading test split from gs://recosys-data-bucket/samples/users_sample_500k/test/ ...
  test_df        : (3451452, 9)

Loading checkpoint: epoch_30.pt ...
  Epoch      : 30
  Train loss : 4.2533

  PART A — TRAINING SCORE DISTRIBUTION
  5 batches × 2,048 pairs  (raw cosine sim, pre-temperature)
  Sampling 5 batches of 2,048 pairs from 7,473,130 training pairs ...
    batch 1/5  |  pos mean: 0.4184  |  neg mean: 0.0222
    batch 2/5  |  pos mean: 0.4180  |  neg mean: 0.0240
    batch 3/5  |  pos mean: 0.4188  |  neg mean: 0.0234
    batch 4/5  |  pos mean: 0.4146  |  neg mean: 0.0218
    batch 5/5  |  pos mean: 0.4139  |  neg mean: 0.0235

  ---------------------------

Full evaluation with popularity collapse and per-user diagnostics on both checkpoints

In [ ]:
!pip install faiss-cpu --quiet
!mkdir -p /content/RecoSys/secrets/
!cp /content/recosys-service-account.json /content/RecoSys/secrets/recosys-service-account.json
!sed -i 's/checkpoints_v2/checkpoints/g' scripts/two_tower/evaluate_two_tower.py
!python scripts/two_tower/evaluate_two_tower.py

Loading artifacts...
  items_encoded  : (284523, 9)
  users_encoded  : (445150, 12)
  train_pairs    : (7473130, 3)
Loading test split from gs://recosys-data-bucket/samples/users_sample_500k/test/ ...
  test_df        : (3451452, 9)
  device         : cuda

Evaluating checkpoint: epoch_5.pt
  train loss at epoch 5: 5.0083
USER COVERAGE DIAGNOSTIC
  Step 1 | Unique users with cart/purchase in test split :  52,402
  Step 2 | Of those, in vocabs['user2idx']               :  39,516  (lost: 12,886 cold-start Feb users)
  Step 3 | Of those, in users_encoded_df                 :  39,516  (lost: 0 users in vocab but missing feature row)
  Step 4 | Eligible eval pool (final)                    :  39,516
  (users_encoded_df unique user_id count                 : 445,150)
  (users_encoded_df unique user_idx count                : 445,150)
Auto-derived trained_item_idxs: 201,648 items
Building item index with 201,648 trained items only (filtered from 284,523 total)
Item index ready: 201,648 items,

In [ ]:
# The script automatically evaluates all downloaded checkpoints listed in EVAL_EPOCHS.
# No need to pass --checkpoint.
!python scripts/two_tower/evaluate_two_tower.py

Loading artifacts...
  items_encoded  : (284523, 9)
  users_encoded  : (445150, 12)
  train_pairs    : (7473130, 3)
Loading test split from gs://recosys-data-bucket/samples/users_sample_500k/test/ ...
  test_df        : (3451452, 9)
  device         : cuda

Evaluating checkpoint: epoch_5.pt
  train loss at epoch 5: 5.0083
USER COVERAGE DIAGNOSTIC
  Step 1 | Unique users with cart/purchase in test split :  52,402
  Step 2 | Of those, in vocabs['user2idx']               :  39,516  (lost: 12,886 cold-start Feb users)
  Step 3 | Of those, in users_encoded_df                 :  39,516  (lost: 0 users in vocab but missing feature row)
  Step 4 | Eligible eval pool (final)                    :  39,516
  (users_encoded_df unique user_id count                 : 445,150)
  (users_encoded_df unique user_idx count                : 445,150)
Auto-derived trained_item_idxs: 201,648 items
Building item index with 201,648 trained items only (filtered from 284,523 total)
Item index ready: 201,648 items,

## Hard Negatives Training:

In [8]:
!ls

artifacts  notebooks  reports		scripts
LICENSE    README.md  requirements.txt	src


In [14]:
%cd /content/RecoSys

import sys
if '/content/RecoSys' not in sys.path:
    sys.path.insert(0, '/content/RecoSys')

from src.two_tower.data.dataset import TwoTowerDatasetWithHardNegs
import pandas as pd, pickle

items = pd.read_parquet('artifacts/500k/items_encoded.parquet')
users = pd.read_parquet('artifacts/500k/users_encoded.parquet')
pairs = pd.read_parquet('artifacts/500k/train_pairs.parquet')
with open('artifacts/500k/vocabs.pkl', 'rb') as f:
    vocabs = pickle.load(f)

ds = TwoTowerDatasetWithHardNegs(pairs, users, items)
print(ds)

/content/RecoSys
TwoTowerDatasetWithHardNegs(pairs=7,473,130, cat_l2=4,885,391, price_bucket_fallback=2,587,739, ratio=[65.4% cat_l2 / 34.6% price_bucket], hard_negs_per_pair=3)


In [ ]:
import time
n = 200_000
t0 = time.time()
_ = TwoTowerDatasetWithHardNegs(pairs.head(n), users, items)
dt = time.time() - t0
print(f"{dt:.1f}s for {n:,} pairs")
print(f"Estimated full time: {dt * len(pairs) / n / 60:.1f} minutes")

In [21]:
!pip install -q faiss-cpu
!cd /content/RecoSys   # your repo root

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 85.0 MB/s eta 0:00:00


In [18]:
import os, subprocess
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/recosys-service-account.json"

In [24]:
!python scripts/two_tower/train_two_tower_v2.py

Loading artifacts...
  items_encoded  : (284523, 9)
  users_encoded  : (445150, 12)
  train_pairs    : (7473130, 3)
Loading test split from gs://recosys-data-bucket/samples/users_sample_500k/test/ ...
  test_df        : (3451452, 9)
  device         : cuda
  hard negatives : True  (ratio: 3)
TwoTowerModel — parameter summary
  User tower :   14,280,504
  Item tower :    9,224,864
  Total      :   23,505,368
  Loaded hard-negative cache: /content/RecoSys/artifacts/500k/hard_neg_idxs_seed42.npy  shape=(7473130, 3)
TwoTowerDatasetWithHardNegs(pairs=7,473,130, cat_l2=4,885,391, price_bucket_fallback=2,587,739, ratio=[65.4% cat_l2 / 34.6% price_bucket], hard_negs_per_pair=3)
  Batches/epoch : 3,649

Training on cuda — 40 epochs, batch 2048, lr 0.001 → 1e-5 (cosine), wd 1e-05

Epoch 1/40
  batch 100/3649 — loss: 7.3710
  batch 200/3649 — loss: 7.1844
  batch 300/3649 — loss: 7.0712
  batch 400/3649 — loss: 6.9723
  batch 500/3649 — loss: 6.9097
  batch 600/3649 — loss: 6.8339
  batch 700/364

In [ ]:
# --- configure ---------------------------------------------------------------
BUCKET_NAME = "recosys-data-bucket"
GCS_PREFIX  = "models/two_tower_500k_v2/"   # new folder; trailing slash ok
LOCAL_DIR   = "artifacts/500k/checkpoints_v2"
# -----------------------------------------------------------------------------

import os
from pathlib import Path

from google.cloud import storage

def upload_file(client: storage.Client, bucket_name: str, local_path: Path, blob_path: str) -> None:
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(blob_path)
    blob.upload_from_filename(str(local_path))
    print(f"  gs://{bucket_name}/{blob_path}  <=  {local_path}")

root = Path(LOCAL_DIR)
if not root.is_dir():
    raise FileNotFoundError(f"Missing {root} — run training first or fix path")

# Uses GOOGLE_APPLICATION_CREDENTIALS (same as training) or gcloud ADC
client = storage.Client()

files = sorted(root.iterdir(), key=lambda p: p.name)
# upload checkpoints + log; skip stray dirs
to_upload = [p for p in files if p.is_file() and (p.suffix in {".pt", ".json"} or p.name == "training_log.json")]

if not to_upload:
    raise RuntimeError(f"No .pt or training_log.json under {root}")

print(f"Uploading {len(to_upload)} file(s) to gs://{BUCKET_NAME}/{GCS_PREFIX}\n")
for p in to_upload:
    rel = f"{GCS_PREFIX.rstrip('/')}/{p.name}"
    upload_file(client, BUCKET_NAME, p, rel)

print("\nDone.")

Different Hard Mining:

In [ ]:
!python scripts/two_tower/train_two_tower_v2.py